In [1]:
pip install statsmodels --break-system-packages


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_csv('../data_processed/shipsense_features.csv')
print(df.shape)

(96478, 14)


In [3]:
customer_summary = df.groupby('customer_unique_id').agg(
    total_orders=('order_id', 'count'),
    ever_late=('is_late', 'max')  # 1 if any of their orders was late
).reset_index()

customer_summary['is_repeat'] = (customer_summary['total_orders'] > 1).astype(int)
print(customer_summary['ever_late'].value_counts())
print(customer_summary['is_repeat'].value_counts())

ever_late
0    86860
1     6498
Name: count, dtype: int64
is_repeat
0    90557
1     2801
Name: count, dtype: int64


In [4]:
# Sort by purchase time so we can identify each customer's FIRST order
df_sorted = df.sort_values('purchase_ts')

first_orders = df_sorted.groupby('customer_unique_id').first().reset_index()
order_counts = df.groupby('customer_unique_id')['order_id'].count().reset_index(name='total_orders')

customer_summary = first_orders.merge(order_counts, on='customer_unique_id')
customer_summary['is_repeat'] = (customer_summary['total_orders'] > 1).astype(int)
customer_summary = customer_summary.rename(columns={'is_late': 'first_order_late'})

print(customer_summary['first_order_late'].value_counts())
print(customer_summary['is_repeat'].value_counts())

first_order_late
0    87003
1     6355
Name: count, dtype: int64
is_repeat
0    90557
1     2801
Name: count, dtype: int64


In [5]:
group_late = customer_summary[customer_summary['first_order_late'] == 1]
group_ontime = customer_summary[customer_summary['first_order_late'] == 0]

count = [group_late['is_repeat'].sum(), group_ontime['is_repeat'].sum()]
nobs = [len(group_late), len(group_ontime)]

z_stat, p_value = proportions_ztest(count, nobs)

repeat_rate_late = group_late['is_repeat'].mean() * 100
repeat_rate_ontime = group_ontime['is_repeat'].mean() * 100

print(f"Repeat rate (first order was late): {repeat_rate_late:.2f}%")
print(f"Repeat rate (first order on time): {repeat_rate_ontime:.2f}%")
print(f"Z-statistic: {z_stat:.3f}")
print(f"P-value: {p_value:.5f}")

Repeat rate (first order was late): 2.52%
Repeat rate (first order on time): 3.04%
Z-statistic: -2.336
P-value: 0.01949


In [6]:
from statsmodels.stats.proportion import confint_proportions_2indep

ci_low, ci_high = confint_proportions_2indep(
    count1=group_late['is_repeat'].sum(), nobs1=len(group_late),
    count2=group_ontime['is_repeat'].sum(), nobs2=len(group_ontime),
    method='wald'
)
print(f"95% CI for the difference in repeat rate: [{ci_low*100:.2f}%, {ci_high*100:.2f}%]")

95% CI for the difference in repeat rate: [-0.92%, -0.12%]


## Hypothesis Test: Does Late Delivery Reduce Repeat Purchases?

**H0**: Repeat-purchase rate is the same for customers whose first order 
arrived late vs. those whose first order arrived on time.

**Result**: Customers whose first order was late had a repeat rate of 2.52%, 
vs 3.04% for customers whose first order arrived on time (p = 0.0195, 
95% CI: [-0.92%, -0.12%]).

**Conclusion**: The difference is statistically significant (p < 0.05) — 
late delivery is associated with a measurably lower repeat-purchase rate. 
However, the effect size is small in absolute terms (about half a 
percentage point), so while the signal is real, delivery delay alone is 
not the dominant driver of repeat-purchase behavior — it's one contributing 
factor among several (price, product category, customer satisfaction, etc.) 
that a fuller model would need to account for.